In [11]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense
from sklearn.model_selection import train_test_split

# ======================
# Load Dataset
# ======================

df = pd.read_csv("stock_data_5yrs.csv")

# Convert Date
df['Date'] = pd.to_datetime(df['Date'])
# Remove timezone
df['Date'] = df['Date'].dt.tz_localize(None)

# ======================
# Encode Company Names
# ======================

le = LabelEncoder()
df['Company'] = le.fit_transform(df['Company'])

# ======================
# Date Features
# ======================

df['Days'] = (df['Date'] - df['Date'].min()).dt.days

# ======================
# Clean Dataset
# ======================

df.replace([np.inf, -np.inf], np.nan, inplace=True)

df.dropna(inplace=True)

# ======================
# Features and Target
# ======================

feature_cols = [col for col in df.columns if col not in ['Close','Date']]

X = df[feature_cols].values
y = df['Close'].values

# ======================
# Scaling
# ======================

scalerX = MinMaxScaler()
scalerY = MinMaxScaler()

X = scalerX.fit_transform(X)
y = scalerY.fit_transform(y.reshape(-1,1))

# ======================
# Reshape for LSTM
# ======================

X = X.reshape((X.shape[0],1,X.shape[1]))

# ======================
# Train Test Split
# ======================

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,shuffle=False)

# ======================
# LSTM Model
# ======================

model = Sequential()

model.add(LSTM(64,input_shape=(1,X.shape[2])))
model.add(Dense(32))
model.add(Dense(1))

model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

model.summary()

# ======================
# Train Model
# ======================

model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32
)

# ======================
# User Input Prediction
# ======================

company_name = input("Enter Company Name: ")
date_input = input("Enter Date (YYYY-MM-DD): ")

date_input = pd.to_datetime(date_input).tz_localize(None)

# Filter company data
company_df = df[df['Company'] == le.transform([company_name])[0]]

# Take last row as base
last_row = company_df.iloc[-1:].copy()

# Update date
last_row['Days'] = (date_input - df['Date'].min()).days
last_row['DayOfWeek'] = date_input.dayofweek
last_row['Month'] = date_input.month

# Prepare input
X_input = last_row[feature_cols].values

X_input = scalerX.transform(X_input)

X_input = X_input.reshape((1,1,X_input.shape[1]))

# Predict
predicted_price = model.predict(X_input)

predicted_price = scalerY.inverse_transform(predicted_price)

print("\nPredicted Close Price:",predicted_price[0][0])

c:\Users\FLEMIN P DANIEL\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        23,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,409 (99.25 KB)

 Trainable params: 25,409 (99.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 7s 3ms/step - loss: 2.7552e-04
Epoch 2/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 5.8257e-06
Epoch 3/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 8.9576e-06
Epoch 4/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 5.8582e-06
Epoch 5/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 5.7021e-06
Epoch 6/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 1.0824e-05
Epoch 7/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 7.2403e-06
Epoch 8/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 3.4404e-06
Epoch 9/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 4.7900e-06
Epoch 10/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 3.6836e-06
Epoch 11/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 3.3612e-06
Epoch 12/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 3.3652e-06
Epoch 13/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 3.4938e-06
Epoch 14/20
1453/1453 ━━━━━━━━━━━━━━━━━━━━ 4s 3

In [12]:
# =========================
# VIF (Multicollinearity Check)
# =========================

from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np

# Select only numeric features (exclude target Close if predicting Close)
vif_df = df.copy()

# Drop non-numeric columns
vif_df = vif_df.select_dtypes(include=[np.number])

# Remove target variable
if 'Close' in vif_df.columns:
    vif_df = vif_df.drop(columns=['Close'])

# Remove columns with NaN or Inf
vif_df = vif_df.replace([np.inf, -np.inf], np.nan)
vif_df = vif_df.dropna()

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["Feature"] = vif_df.columns
vif_data["VIF"] = [
    variance_inflation_factor(vif_df.values, i)
    for i in range(len(vif_df.columns))
]

# Sort by highest VIF
vif_data = vif_data.sort_values(by="VIF", ascending=False)

print(vif_data)

c:\Users\FLEMIN P DANIEL\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


         Feature           VIF
1           Open           inf
2           High           inf
3            Low           inf
23  TypicalPrice           inf
24       CO_Diff           inf
15         Range           inf
21         EMA10  2.710746e+06
22         EMA20  5.925647e+05
9           MA10  2.970853e+05
8            MA5  2.246293e+05
10          MA20  1.278281e+05
11          MA50  1.524768e+04
25          Days  5.482321e+02
0     Unnamed: 0  5.481720e+02
13      Momentum  9.549517e+00
17      Vol_MA10  4.623109e+00
4         Volume  4.372315e+00
16    Volatility  4.259990e+00
7        Company  3.371534e+00
20         Month  3.361666e+00
19     DayOfWeek  2.576464e+00
14           ROC  1.657851e+00
12        Return  1.567932e+00
18    Vol_Change  1.142584e+00
5      Dividends  1.004397e+00
6   Stock Splits  1.000922e+00
